In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("⚽ Iniciando Sistema de Inteligencia Deportiva (Versión IA Pura)...")

# ==========================================
# 1. CONEXIÓN Y CARGA DE DATOS
# ==========================================
print("-> Extrayendo datos...")
conn = sqlite3.connect('database.sqlite')
df_names = pd.read_sql("SELECT team_api_id, team_long_name FROM Team", conn)

query_p = """
SELECT p.player_name, p.birthday, pa.* FROM Player_Attributes pa 
JOIN Player p ON pa.player_api_id = p.player_api_id 
ORDER BY pa.date DESC
"""
df_players = pd.read_sql_query(query_p, conn)
df_players = df_players.drop_duplicates(subset=['player_api_id'], keep='first').dropna()

# ==========================================
# 2. BLOQUE JUGADORES: IA Y ROLES
# ==========================================
print("-> Entrenando Red de Scouting...")
df_players['date'] = pd.to_datetime(df_players['date'])
df_players['birthday'] = pd.to_datetime(df_players['birthday'])
df_players['edad'] = (df_players['date'] - df_players['birthday']).dt.days // 365

df_outfield = df_players[df_players['gk_diving'] <= 40].copy()
features = ['reactions', 'ball_control', 'short_passing', 'long_passing', 'shot_power', 'vision', 'dribbling', 'long_shots', 'curve']
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(df_outfield[features], df_outfield['overall_rating'])
df_outfield['valoracion_tecnica'] = model.predict(df_outfield[features])

df_outfield['rol_principal'] = df_outfield.apply(lambda r: max({
    'Creador': r['short_passing']*0.5 + r['vision']*0.5,
    'Finalizador': r['finishing']*0.6 + r['shot_power']*0.4,
    'Defensivo': r['marking']*0.5 + r['standing_tackle']*0.5
}, key=lambda k: {
    'Creador': r['short_passing']*0.5 + r['vision']*0.5,
    'Finalizador': r['finishing']*0.6 + r['shot_power']*0.4,
    'Defensivo': r['marking']*0.5 + r['standing_tackle']*0.5
}[k]), axis=1)

df_gk = df_players[df_players['gk_diving'] > 40].copy()
df_gk['valoracion_tecnica'] = df_gk[['gk_diving', 'gk_handling', 'gk_kicking', 'gk_positioning', 'gk_reflexes']].mean(axis=1)
df_gk['rol_principal'] = 'Portero'

df_f = pd.concat([df_outfield, df_gk])

df_f['potencial_ia'] = df_f.apply(lambda r: r['valoracion_tecnica'] + ((28 - r['edad']) * (2.0 if r['edad'] < 19 and r['valoracion_tecnica'] > 60 else 1.2)) if r['edad'] < 28 else r['valoracion_tecnica'], axis=1)
df_f['ventaja_ia'] = df_f['valoracion_tecnica'] - df_f['overall_rating']
df_f['crecimiento_proyectado'] = df_f['potencial_ia'] - df_f['valoracion_tecnica']
raw_p = (df_f['ventaja_ia'] * 0.6) + (df_f['crecimiento_proyectado'] * 0.4)
df_f['prioridad_fichaje'] = (((raw_p - raw_p.min()) / (raw_p.max() - raw_p.min())) * 100).round(1)

# ==========================================
# 2.5 RASTREADOR GLOBAL DE EQUIPOS
# ==========================================
print("-> Rastreador Global: Mapeando equipos reales...")
cols_jugadores = [f'home_player_{i}' for i in range(1, 12)] + [f'away_player_{i}' for i in range(1, 12)]
query_global = f"SELECT date, home_team_api_id, away_team_api_id, {', '.join(cols_jugadores)} FROM Match"
df_global_matches = pd.read_sql(query_global, conn)

h_melt = df_global_matches.melt(id_vars=['date', 'home_team_api_id'], value_vars=[f'home_player_{i}' for i in range(1, 12)], value_name='player_api_id').dropna()
h_melt = h_melt.rename(columns={'home_team_api_id': 'team_api_id'})
a_melt = df_global_matches.melt(id_vars=['date', 'away_team_api_id'], value_vars=[f'away_player_{i}' for i in range(1, 12)], value_name='player_api_id').dropna()
a_melt = a_melt.rename(columns={'away_team_api_id': 'team_api_id'})

all_players_teams = pd.concat([h_melt, a_melt])
all_players_teams['date'] = pd.to_datetime(all_players_teams['date'])
latest_team = all_players_teams.sort_values('date').drop_duplicates(subset=['player_api_id'], keep='last')
latest_team = pd.merge(latest_team, df_names, on='team_api_id', how='left')
latest_team = latest_team[['player_api_id', 'team_long_name']].rename(columns={'team_long_name': 'Equipo Actual'})

df_f = pd.merge(df_f, latest_team, on='player_api_id', how='left')
df_f['Equipo Actual'] = df_f['Equipo Actual'].fillna('Retirado / Agente Libre Real')

# CSV 1 con Calidad IA
df_jugadores_final = df_f[['player_name', 'overall_rating', 'edad', 'Equipo Actual', 'rol_principal', 'valoracion_tecnica', 'potencial_ia', 'prioridad_fichaje']].rename(columns={
    'player_name': 'Nombre Jugador', 'overall_rating': 'Media Oficial', 'edad': 'Edad', 'rol_principal': 'Rol Táctico', 
    'valoracion_tecnica': 'Calidad IA', 'potencial_ia': 'Potencial IA', 'prioridad_fichaje': 'Índice de Prioridad (%)'
})

# ==========================================
# 3. BLOQUE ESTRATEGIA Y PRESTIGIO
# ==========================================
print("-> Calculando Prestigio y Gravedad de Goles...")
query_m = "SELECT season, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM Match WHERE league_id = 21518"
df_m = pd.read_sql(query_m, conn)

df_m['h_pts'] = np.where(df_m['home_team_goal'] > df_m['away_team_goal'], 3, np.where(df_m['home_team_goal'] == df_m['away_team_goal'], 1, 0))
df_m['a_pts'] = np.where(df_m['away_team_goal'] > df_m['home_team_goal'], 3, np.where(df_m['away_team_goal'] == df_m['home_team_goal'], 1, 0))
p_h = df_m.groupby(['season', 'home_team_api_id'])['h_pts'].sum().reset_index().rename(columns={'home_team_api_id':'team_id', 'h_pts':'puntos'})
p_a = df_m.groupby(['season', 'away_team_api_id'])['a_pts'].sum().reset_index().rename(columns={'away_team_api_id':'team_id', 'a_pts':'puntos'})
df_puntos = pd.concat([p_h, p_a]).groupby(['season', 'team_id'])['puntos'].sum().reset_index()

df_puntos_nombres = pd.merge(df_puntos, df_names, left_on='team_id', right_on='team_api_id')
df_prestigio = df_puntos_nombres.groupby('team_long_name')['puntos'].mean().reset_index().rename(columns={'puntos': 'puntos_medios'})
df_prestigio = df_prestigio.sort_values('puntos_medios', ascending=False)

g_h = df_m.groupby('home_team_api_id').agg(gf_h=('home_team_goal', 'sum'), gc_h=('away_team_goal', 'sum'), p_h=('home_team_api_id', 'count')).reset_index().rename(columns={'home_team_api_id': 'team_id'})
g_v = df_m.groupby('away_team_api_id').agg(gf_v=('away_team_goal', 'sum'), gc_v=('home_team_goal', 'sum'), p_v=('away_team_api_id', 'count')).reset_index().rename(columns={'away_team_api_id': 'team_id'})
df_goles = pd.merge(g_h, g_v, on='team_id').fillna(0)
df_goles['goles_favor_partido'] = (df_goles['gf_h'] + df_goles['gf_v']) / (df_goles['p_h'] + df_goles['p_v'])
df_goles['goles_contra_partido'] = (df_goles['gc_h'] + df_goles['gc_v']) / (df_goles['p_h'] + df_goles['p_v'])
df_goles_nombres = pd.merge(df_goles, df_names, left_on='team_id', right_on='team_api_id')

df_t_attr = pd.read_sql("SELECT * FROM Team_Attributes", conn).rename(columns={'team_api_id': 'team_id'})
df_t_attr['year'] = pd.to_datetime(df_t_attr['date']).dt.year.astype(str)
df_puntos['year'] = df_puntos['season'].astype(str).str[:4]
df_strat = pd.merge(df_puntos, df_t_attr, on=['year', 'team_id'])
df_strat_final = pd.merge(df_strat, df_names, left_on='team_id', right_on='team_api_id')

dicc_valores = {'Slow': 'Lento', 'Balanced': 'Equilibrado', 'Fast': 'Rápido', 'Little': 'Poco', 'Normal': 'Normal', 'Lots': 'Mucho', 'Short': 'Corto', 'Mixed': 'Mixto', 'Long': 'Largo', 'Organised': 'Organizado', 'Free Form': 'Libre', 'Risky': 'Arriesgado', 'Deep': 'Baja', 'Medium': 'Media', 'High': 'Alta', 'Contain': 'Contener', 'Press': 'Presionar', 'Double': 'Doble', 'Cover': 'Cobertura', 'Offside Trap': 'Fuera de Juego', 'Safe': 'Seguro', 'Wide': 'Ancha', 'Narrow': 'Estrecha'}
df_strat_final = df_strat_final.replace(dicc_valores)
trad_cols_strat = {'season': 'Temporada', 'puntos': 'Puntos', 'team_long_name': 'Equipo', 'buildUpPlaySpeed': 'Velocidad Juego', 'buildUpPlaySpeedClass': 'Tipo Velocidad', 'buildUpPlayDribbling': 'Nivel Regate', 'buildUpPlayDribblingClass': 'Tipo Regate', 'buildUpPlayPassing': 'Estilo Pase', 'buildUpPlayPassingClass': 'Tipo Pase', 'buildUpPlayPositioningClass': 'Posicionamiento Salida', 'chanceCreationPassing': 'Pase Creación', 'chanceCreationPassingClass': 'Tipo Pase Creación', 'chanceCreationCrossing': 'Frecuencia Centros', 'chanceCreationCrossingClass': 'Tipo Centros', 'chanceCreationShooting': 'Volumen Tiros', 'chanceCreationShootingClass': 'Tipo Tiros', 'chanceCreationPositioningClass': 'Posicionamiento Ataque', 'defencePressure': 'Nivel Presión', 'defencePressureClass': 'Tipo Presión', 'defenceAggression': 'Agresividad', 'defenceAggressionClass': 'Tipo Agresividad', 'defenceTeamWidth': 'Anchura Bloque', 'defenceTeamWidthClass': 'Tipo Anchura', 'defenceDefenderLineClass': 'Línea Defensiva'}
df_strat_final = df_strat_final.rename(columns=trad_cols_strat)

if 'Nivel Regate' in df_strat_final.columns:
    media_regate = df_strat_final['Nivel Regate'].mean()
    if pd.isna(media_regate): media_regate = 50.0 
    df_strat_final['Nivel Regate'] = df_strat_final['Nivel Regate'].fillna(media_regate).round(1)

# ==========================================
# 4. MOTOR DE DIRECTOR DEPORTIVO (DRAFT)
# ==========================================
print("-> Ejecutando Scouting con Nuestra Valoración Técnica...")

jugadores_fichados = set()

def diagnostico_y_recomendacion(equipo, df_jugadores, df_goles_stats, df_tacticas, df_prestigio):
    global jugadores_fichados
    stats_equipo = df_goles_stats[df_goles_stats['team_long_name'] == equipo]
    tactica = df_tacticas[df_tacticas['Equipo'] == equipo].iloc[0]
    puntos_club = df_prestigio[df_prestigio['team_long_name'] == equipo]['puntos_medios'].values[0]
    
    if stats_equipo.empty: return []
    gf = stats_equipo['goles_favor_partido'].values[0]
    gc = stats_equipo['goles_contra_partido'].values[0]
    
    # Prestigio basado en Puntos
    if puntos_club >= 75: tope_ia = 99
    elif puntos_club >= 60: tope_ia = 82
    elif puntos_club >= 45: tope_ia = 78
    else: tope_ia = 74

    # Gravedad Relativa
    gravedad_defensa = gc - 1.35 
    gravedad_ataque = 1.35 - gf  
    
    if gravedad_defensa > gravedad_ataque and gravedad_defensa > 0:
        urgencia = f"Coladero en Defensa (Recibe {gc:.2f} p/p)"
        rol_necesario = "Defensivo"
    elif gravedad_ataque > gravedad_defensa and gravedad_ataque > 0:
        urgencia = f"Falta de Gol (Marca {gf:.2f} p/p)"
        rol_necesario = "Finalizador"
    else:
        urgencia = "Rendimiento Equilibrado"
        rol_necesario = "Creador"
        
    mercado = df_jugadores[df_jugadores['rol_principal'] == rol_necesario].copy()
    
    # Filtros: Ahora usamos NUESTRA valoración técnica para el tope de mercado
    mercado = mercado[mercado['valoracion_tecnica'] <= tope_ia]
    mercado = mercado[~mercado['player_name'].isin(jugadores_fichados)]
    mercado = mercado[mercado['Equipo Actual'] != equipo]
    
    if mercado.empty: return []

    mercado['Nota_Base'] = mercado['valoracion_tecnica'] * 0.3
    
    if rol_necesario == 'Finalizador':
        if tactica.get('Tipo Centros', '') == 'Mucho':
            encaje = (mercado['jumping'] * 0.5) + (mercado['strength'] * 0.5)
            perfil = "Delantero Tanque"
        else:
            encaje = (mercado['sprint_speed'] * 0.5) + (mercado['finishing'] * 0.5)
            perfil = "Finalizador Rápido"
    elif rol_necesario == 'Defensivo':
        if tactica.get('Tipo Presión', '') == 'Alta':
            encaje = (mercado['sprint_speed'] * 0.5) + (mercado['aggression'] * 0.5)
            perfil = "Central Rápido"
        else:
            encaje = (mercado['marking'] * 0.5) + (mercado['strength'] * 0.5)
            perfil = "Central Físico"
    else:
        encaje = (mercado['vision'] * 0.5) + (mercado['short_passing'] * 0.5)
        perfil = "Organizador Técnico"

    mercado['Penalizacion_Edad'] = np.where(mercado['edad'] > 32, (mercado['edad'] - 32) * 3, 0)
    mercado['Match_Total (%)'] = (mercado['Nota_Base'] + (encaje * 0.7)) - mercado['Penalizacion_Edad']
    mercado['Match_Total (%)'] = mercado['Match_Total (%)'].clip(lower=0, upper=100.0).round(1)
    
    top_5 = mercado.sort_values('Match_Total (%)', ascending=False).head(5)
    jugadores_fichados.update(top_5['player_name'].tolist())
    
    resultados = []
    for _, player in top_5.iterrows():
        resultados.append({
            'Equipo Destino': equipo,
            'Alarma Detectada': urgencia,
            'Rol Táctico Exigido': perfil,
            'Jugador Recomendado': player['player_name'],
            'Procedencia (Equipo Actual)': player['Equipo Actual'],
            'Edad': player['edad'],
            # CAMBIO CLAVE: Sustituimos Media Oficial por Nuestra Valoración Técnica
            'Nuestra valoración técnica': round(float(player['valoracion_tecnica']), 1),
            'Afinidad Táctica Total (%)': float(player['Match_Total (%)'])
        })
    return resultados

# ==========================================
# 5. EXPORTACIÓN
# ==========================================
print("-> Escribiendo CSVs finales...")

equipos_ordenados = df_prestigio['team_long_name'].tolist()
informe_director_deportivo = []

for equipo in equipos_ordenados:
    if equipo in df_strat_final['Equipo'].values:
        try:
            recomendaciones = diagnostico_y_recomendacion(equipo, df_f, df_goles_nombres, df_strat_final, df_prestigio)
            informe_director_deportivo.extend(recomendaciones)
        except: pass

df_recomendador_final = pd.DataFrame(informe_director_deportivo)

# Limpieza CSV 2
columnas_borrar = [c for c in ['team_id', 'team_api_id', 'date', 'id', 'team_fifa_api_id', 'year'] if c in df_strat_final.columns]
df_strat_final = df_strat_final.drop(columns=columnas_borrar)

conn.close()

df_jugadores_final.to_csv('Jugadores_Stats.csv', index=False, encoding='utf-8-sig')
df_strat_final.to_csv('Partidos_Stats.csv', index=False, encoding='utf-8-sig')
df_recomendador_final.to_csv('Recomendaciones_Fichajes.csv', index=False, encoding='utf-8-sig')

print("🚀")

⚽ Iniciando Sistema de Inteligencia Deportiva (Versión IA Pura)...
-> Extrayendo datos...
-> Entrenando Red de Scouting...
-> Rastreador Global: Mapeando equipos reales...
-> Calculando Prestigio y Gravedad de Goles...
-> Ejecutando Scouting con Nuestra Valoración Técnica...
-> Escribiendo CSVs finales...
🚀
